# Decision making and diffusion experiments
Extract the updated GPU bundle on your Windows NVIDIA machine. Select the CUDA-enabled PyTorch kernel in VS Code.

This notebook adds:
1. **Offline action prediction**: XLM-R predicts the source `decision_class` (13 actions) from conversation text.
2. **Conditional label diffusion**: a CARD-inspired denoiser predicts risk labels or actions, compared with a linear SVM and MLP on the same train-fitted TF-IDF/SVD features.

The diffusion experiment generates **labels**, not conversation text. It is an adaptation of [CARD](https://arxiv.org/abs/2206.07275), not a reproduction. See `DECISIONS_AND_DIFFUSION.md` for the equations, protocol and limits. No actions are executed.

The original risk fine-tuning experiment is still in notebook 05. Run this notebook separately; its outputs use different directories.

In [ ]:
%pip install "transformers==4.57.6" "accelerate>=1.1,<2" "sentencepiece>=0.2,<0.3" "scikit-learn>=1.6,<1.8"

In [ ]:
import json
import pathlib
import subprocess
import sys
import torch

ROOT = pathlib.Path.cwd()
assert (ROOT / "manifest.json").exists(), "Open the notebook from the extracted bundle folder."
manifest = json.loads((ROOT / "manifest.json").read_text())
assert "decision_classes" in manifest, "This is the older bundle. Transfer the updated ZIP."
assert torch.cuda.is_available(), "Select a kernel with CUDA-enabled PyTorch."
print(torch.cuda.get_device_name(0))
print("Actions:", manifest["decision_classes"])
for name, counts in manifest["counts"].items():
    print(name, counts["decisions"])

OUTPUT = ROOT / "decision_diffusion_results"
OUTPUT.mkdir(exist_ok=True)
SEED = 42
BATCH_SIZE = 2
ACCUMULATION = 16

def run(script, *args):
    subprocess.run([sys.executable, "-u", str(ROOT / script), *map(str, args)], cwd=ROOT, check=True)

## Smoke checks
These exercise forward/backward passes, checkpoint saving and validation without gold evaluation. If the XLM-R check runs out of CUDA memory, set `BATCH_SIZE=1` and `ACCUMULATION=32`. Existing outputs are protected: choose a fresh directory when rerunning a smoke check.

In [ ]:
run("train_transformer.py", "train", "--data", ROOT, "--target", "action",
    "--variant", "mixed", "--output", OUTPUT / "action_xlmr_smoke",
    "--batch-size", BATCH_SIZE, "--accumulation", ACCUMULATION, "--smoke")
run("run_label_diffusion.py", "train", "--data", ROOT, "--target", "action",
    "--output", OUTPUT / "diffusion_smoke", "--smoke")

## Train action prediction
XLM-R learns the original agent action rather than our derived risk class. Only message content is tokenized. The target, recorded response, explanation and diagnostic fields are excluded from input. Both XLM-R targets use the same family split and common EN + silver development data.

Three epochs and the same optimizer settings as notebook 05. Per-class counts above reveal rare actions; some may not occur in the 300-pair evaluation.

In [ ]:
ACTION_MODEL = OUTPUT / f"action_xlmr_mixed_seed{SEED}"
run("train_transformer.py", "train", "--data", ROOT, "--target", "action",
    "--variant", "mixed", "--output", ACTION_MODEL, "--seed", SEED,
    "--batch-size", BATCH_SIZE, "--accumulation", ACCUMULATION)

## Train conditional diffusion and matched baselines
Each run fits text features on training examples only, trains an MLP conditional mean, freezes its validation-selected checkpoint, and trains a noise-prediction network on one-hot label residuals.

Default: 128 SVD dimensions, hidden size 256, 30 epochs per neural head, 100 diffusion steps, eight samples per prediction. Select diffusion checkpoints on the same development data every five epochs. The linear SVM, MLP and diffusion head all receive identical dense text features.

Start with the mixed-data condition. A later English-only experiment uses `--variant en` and a fresh output directory, with other settings fixed. Development still includes silver Kazakh in both conditions.

In [ ]:
for target in ["risk", "action"]:
    run("run_label_diffusion.py", "train", "--data", ROOT, "--target", target,
        "--variant", "mixed", "--output", OUTPUT / f"diffusion_{target}_mixed_seed{SEED}",
        "--seed", SEED)

In [ ]:
for folder in [ACTION_MODEL,
               OUTPUT / f"diffusion_risk_mixed_seed{SEED}",
               OUTPUT / f"diffusion_action_mixed_seed{SEED}"]:
    print(folder.name)
    print(json.loads((folder / "validation_metrics.json").read_text()))

## Final evaluation
Run after all settings and checkpoints are fixed. This is the historical 300-pair benchmark, which has influenced earlier project decisions. It has not had native-speaker review.

Action evaluation reports fixed-schema macro-F1, macro-F1 over supported classes, accuracy and per-class support. These predict agreement with recorded actions, not real-world safety or successful decision execution.

Risk evaluation retains three-class and merged-binary macro-F1. Diffusion votes show sample agreement, not calibrated confidence. Compare heads on identical features before comparing with end-to-end XLM-R or full TF-IDF. Repeat fixed settings across seeds before claiming an improvement.

In [ ]:
run("train_transformer.py", "evaluate", "--data", ROOT, "--output", ACTION_MODEL,
    "--batch-size", BATCH_SIZE)
for target in ["risk", "action"]:
    run("run_label_diffusion.py", "evaluate", "--data", ROOT,
        "--output", OUTPUT / f"diffusion_{target}_mixed_seed{SEED}")
run("train_transformer.py", "baselines", "--data", ROOT, "--target", "action",
    "--output", OUTPUT / "action_tfidf")

In [ ]:
for target in ["risk", "action"]:
    metrics = json.loads((OUTPUT / f"diffusion_{target}_mixed_seed{SEED}" / "test_metrics.json").read_text())
    print("\n", target)
    for language, values in metrics.items():
        for model in ["linear", "mlp", "diffusion"]:
            m = values[model]
            print(language, model, "macro-F1", round(m["macro_f1"], 4),
                  "supported F1", round(m["macro_f1_supported"], 4),
                  "accuracy", round(m["accuracy"], 4))
        print("Targets absent from training:", values["unseen_training_target_rows"])
        print("Test support:", values["class_support"])

## Bring back results
The ZIP includes predictions, metrics, training logs and the split manifest. Large model and feature files stay on this machine.

In [ ]:
import zipfile
archive = ROOT / "decision_diffusion_summary.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT / "manifest.json", "manifest.json")
    for path in OUTPUT.rglob("*.json"):
        if any(p.startswith("checkpoint-") for p in path.relative_to(OUTPUT).parts):
            continue
        if path.name not in {"tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "config.json"}:
            z.write(path, path.relative_to(ROOT))
print("Bring back:", archive)